# Part 12 - Tools as a Protocol: a minimal MCP server and host by hand

*Agents from First Principles, Part 12.*

**The frontier track begins here.** The core track (Parts 1 to 11) ringed a bare LLM into an agent that can plan, recover, remember, stop itself, survive a crash, ask a human, and be observed. But its tools never left the building. They are still a hardcoded Python dict baked into the same process: `TOOL_SCHEMAS` from Part 1. That dict cannot be shared with another agent, swapped at deploy time, or DISCOVERED by a host that did not import it. Every integration is bespoke wiring.

The **Model Context Protocol (MCP)** fixes this by turning "what tools do you have and how do I call them" into a wire protocol. A SERVER exposes capabilities; a HOST discovers and calls them over JSON-RPC. Once tools speak a protocol, any host can use any server, and the agent's action space is assembled at run time from whatever servers it connects to, not hardcoded at author time.

This part builds both sides by hand, standard library only:

1. **An MCP SERVER.** It answers JSON-RPC requests: the `initialize` handshake (protocolVersion + capability negotiation), `tools/list` (return tool schemas), `tools/call` (run a tool), and the other two of MCP's three server PRIMITIVES: `resources` (read-only data the host can fetch) and `prompts` (named prompt templates). We wrap the support-bot tools as MCP capabilities.

2. **An MCP HOST.** It performs the handshake, calls `tools/list`, and feeds the returned JSON `inputSchema` STRAIGHT INTO Part 1's validator and controller, replacing the hardcoded dict. The same kind of schema object Part 1 hand-wrote now arrives over the wire. And the host MULTIPLEXES: connect to two servers and the agent's palette is the union of both, used transparently.

**Honesty about the transport (this is the whole feasibility question).** The DEFAULT here is an IN-PROCESS JSON-RPC shim: the host calls `server.handle(request)` directly, but every message is a real JSON-RPC frame we print verbatim, so you see the exact bytes that would cross a socket. The real deployment runs the server as a SEPARATE PROCESS the host talks to over stdio. We show that path too, as a LABELED, ILLUSTRATIVE CLI transcript (two local processes, no network), clearly marked as a reference of what it looks like, not a frozen, verified run.

> **Runs with no network, no API key, and no third-party dependency.** The in-process JSON-RPC shim is the offline source of truth; `generate()` (a real LLM controller handed the discovered schemas) and the real stdio subprocess transport are each one flag away. Set `OPENAI_API_KEY` to see the real banner; the code always uses the deterministic shim so output stays reproducible.

Companion script: `mcp_server_and_host.py`

## Setup

Two standard-library imports do the work: `json` serializes every JSON-RPC frame (the exact bytes that would cross a socket), and `os` lets us check for an API key without ever requiring one. Everything else is plain Python, so every cell runs fully offline, exactly as in Parts 1 to 11.

The `protocolVersion` is `2025-06-18`. MCP uses date-style versions and the SDK shapes move fast; check the current spec. Only the transport and the protocolVersion string would ever need edits.

In [ ]:
import json
import os

PROTOCOL_VERSION = "2025-06-18"            # MCP uses date-style versions; check current

print("ready -- no network, no API key, no third-party dependency required")

## Step 1 - The tool implementations (the support-bot world)

These are plain functions, carried from the support-bot world of the RAG series and Parts 1 to 11: a policy search, a products search, and a side-effecting refund (Part 2's point: a refund mutates the world, so it is not idempotent and not free). On MCP these bodies become what runs behind `tools/call`; their hand-written JSON schemas become the `inputSchema` returned by `tools/list`. The functions themselves do not know they are being served over a protocol. That is the whole idea: the wire wraps ordinary functions.

In [ ]:
def _search_policy(query):
    return "Refunds are accepted within 30 days; after the window a 10% restocking fee applies."


def _search_products(query):
    if "acquired" in query.lower():
        return "Acme Corp was acquired by Globex in 2024."
    return "Globex-branded wireless earbuds carry a 2-year limited warranty."


def _process_refund(order_id, amount):
    return f"refunded ${float(amount):.2f} to {order_id}"


print("tool implementations ready (search_policy, search_products, process_refund).")

## Step 2 - The MCP server: a JSON-RPC dispatcher with three primitives

The server is a plain object whose `handle()` dispatches a JSON-RPC request by its `method` and returns a JSON-RPC response. Five methods cover MCP's surface:

- **`initialize`** negotiates the protocol version and advertises which of the three primitives this server supports. A capability is `{}` when present, `None` when absent: the support server offers all three, the catalog server only `tools`.
- **`tools/list`** returns each tool's `name`, `description`, and the typed `inputSchema` (this is what the host discovers and validates against).
- **`tools/call`** runs the named tool with its arguments and wraps the result as MCP content.
- **`resources/read`** returns the text of a read-only resource by URI; **`prompts/get`** returns a named prompt template as messages.

Those last three (`tools`, `resources`, `prompts`) are MCP's **three server primitives**. A clean error response (`-32602` for bad params, `-32603` for anything else) keeps the host from crashing on a malformed call, in the spirit of Part 1's validator and Part 2's failure taxonomy.

In [ ]:
class MCPServer:
    def __init__(self, name, tools, resources, prompts):
        self.name = name
        self.tools = tools                 # name -> {description, inputSchema, fn}
        self.resources = resources         # uri -> {name, text}
        self.prompts = prompts             # name -> {description, template}

    def handle(self, request):
        rid, method, params = request.get("id"), request["method"], request.get("params", {})
        try:
            result = self._dispatch(method, params)
            return {"jsonrpc": "2.0", "id": rid, "result": result}
        except KeyError as exc:
            return {"jsonrpc": "2.0", "id": rid,
                    "error": {"code": -32602, "message": f"invalid params: {exc}"}}
        except Exception as exc:           # defensive
            return {"jsonrpc": "2.0", "id": rid,
                    "error": {"code": -32603, "message": str(exc)}}

    def _dispatch(self, method, params):
        if method == "initialize":
            return {"protocolVersion": PROTOCOL_VERSION,
                    "serverInfo": {"name": self.name},
                    "capabilities": {
                        "tools": {} if self.tools else None,
                        "resources": {} if self.resources else None,
                        "prompts": {} if self.prompts else None}}
        if method == "tools/list":
            return {"tools": [{"name": n, "description": t["description"],
                               "inputSchema": t["inputSchema"]} for n, t in self.tools.items()]}
        if method == "tools/call":
            name, args = params["name"], params.get("arguments", {})
            text = self.tools[name]["fn"](**args)
            return {"content": [{"type": "text", "text": text}], "isError": False}
        if method == "resources/list":
            return {"resources": [{"uri": u, "name": r["name"]} for u, r in self.resources.items()]}
        if method == "resources/read":
            uri = params["uri"]
            return {"contents": [{"uri": uri, "text": self.resources[uri]["text"]}]}
        if method == "prompts/list":
            return {"prompts": [{"name": n, "description": p["description"]}
                                for n, p in self.prompts.items()]}
        if method == "prompts/get":
            name = params["name"]
            return {"messages": [{"role": "user",
                                  "content": {"type": "text", "text": self.prompts[name]["template"]}}]}
        raise Exception(f"method not found: {method}")


print("MCPServer ready (initialize, tools/list, tools/call, resources/*, prompts/*).")

In [ ]:
# Two concrete servers prove discovery and multiplexing are real, not a special
# case. The support server offers ALL THREE primitives: two tools (search_policy,
# process_refund), one resource (the refund policy file), one prompt template. The
# catalog server offers only tools (search_products). The capability difference is
# read off each initialize response, not assumed host-side.
SUPPORT = MCPServer(
    name="support-server",
    tools={
        "search_policy": {
            "description": "search the support/policy index",
            "inputSchema": {"type": "object",
                            "properties": {"query": {"type": "string"}}, "required": ["query"]},
            "fn": _search_policy},
        "process_refund": {
            "description": "issue a refund for an order (side-effecting)",
            "inputSchema": {"type": "object",
                            "properties": {"order_id": {"type": "string"},
                                           "amount": {"type": "number"}},
                            "required": ["order_id", "amount"]},
            "fn": _process_refund},
    },
    resources={"file:///policies/refund.md":
               {"name": "refund-policy", "text": "Refunds within 30 days; 10% restocking fee after."}},
    prompts={"refund_decision":
             {"description": "draft a refund decision",
              "template": "Decide the refund for {order}, citing the policy."}},
)

CATALOG = MCPServer(
    name="catalog-server",
    tools={
        "search_products": {
            "description": "search the products index (acquisitions, warranties)",
            "inputSchema": {"type": "object",
                            "properties": {"query": {"type": "string"}}, "required": ["query"]},
            "fn": _search_products},
    },
    resources={},
    prompts={},
)

print("two servers ready: support-server (tools+resources+prompts), catalog-server (tools).")

## Step 3 - The validator: Part 1's guarantee, now reading a wire-discovered schema

This is Part 1's validator, reused unchanged in spirit. The only difference is where the schema comes from: in Part 1 it was a hardcoded Python dict the author wrote; here it is the JSON `inputSchema` that arrived over `tools/list`. A JSON Schema is just data, so the validator does not care whether it was hand-written or discovered. `validate_against_schema` checks required arguments are present and that each argument's Python type matches the declared JSON type, returning `(ok, error_message)` so a rejection becomes a recoverable observation rather than a crash.

We do not re-derive the validator here; it is the Part 1 reference applied over the wire. The payoff comes in the demo: the exact same guarantee, on a schema the host never hardcoded.

In [ ]:
_PY_TYPE = {"string": str, "number": (int, float), "boolean": bool}


def validate_against_schema(args, input_schema):
    props = input_schema.get("properties", {})
    for req in input_schema.get("required", []):
        if req not in args:
            return False, f"missing required arg '{req}'"
    for k, v in args.items():
        if k not in props:
            return False, f"unexpected arg '{k}'"
        if not isinstance(v, _PY_TYPE[props[k]["type"]]):
            return False, f"arg '{k}' must be {props[k]['type']}"
    return True, None


print("validate_against_schema ready (Part 1's validator, reading a wire-discovered schema).")

## Step 4 - The MCP host: handshake, discovery, and frame printing

The host is the agent's side of the wire. `connect()` performs the `initialize` handshake, prints the negotiated protocol and capabilities, then calls `tools/list` and folds every discovered tool into a single `palette` (tool name -> which server, its schema, its description). `call_tool()` routes a call to whichever server owns that tool. The whole point of `_rpc()` is the two `print` lines: every request and response is dumped verbatim with `json.dumps`, so the in-process shim shows the EXACT bytes a socket would carry.

**The transport is the shim.** `server.handle(req)` is a direct method call, not a socket read. But the framing is real: the dict that goes in and the dict that comes back are valid JSON-RPC messages. Swapping the shim for a real stdio pipe (Step 6) changes only how `req` reaches the server, not a single byte of the frames.

In [ ]:
class MCPHost:
    def __init__(self, verbose=True):
        self.servers = {}
        self.palette = {}                  # tool_name -> {server, schema, description}
        self._id = 0
        self.verbose = verbose

    def _rpc(self, server, method, params):
        self._id += 1
        req = {"jsonrpc": "2.0", "id": self._id, "method": method, "params": params}
        if self.verbose:
            print(f"    --> {json.dumps(req)}")
        resp = server.handle(req)          # in-process shim: a direct call, real frames
        if self.verbose:
            print(f"    <-- {json.dumps(resp)}")
        return resp.get("result", {})

    def connect(self, server):
        init = self._rpc(server, "initialize",
                         {"protocolVersion": PROTOCOL_VERSION,
                          "clientInfo": {"name": "agents-by-hand-host"}})
        caps = [c for c, v in init["capabilities"].items() if v is not None]
        print(f"    [host] connected to {init['serverInfo']['name']} "
              f"(protocol {init['protocolVersion']}, capabilities: {', '.join(caps)})")
        self.servers[server.name] = server
        for t in self._rpc(server, "tools/list", {})["tools"]:
            self.palette[t["name"]] = {"server": server.name, "schema": t["inputSchema"],
                                       "description": t["description"]}

    def call_tool(self, name, arguments):
        server = self.servers[self.palette[name]["server"]]
        res = self._rpc(server, "tools/call", {"name": name, "arguments": arguments})
        return res["content"][0]["text"]


print("MCPHost ready (handshake, discovery, frame printing, multiplexed palette).")

## Step 5 - generate(): the real LLM controller path (reference shape only)

Same device as Parts 1 to 11. On the real path the controller is an LLM, and the tools it is handed are exactly the schemas the host just DISCOVERED over `tools/list`, not a hardcoded list. That is the MCP payoff for the controller: its action space is assembled at run time from whatever servers it connected to. `generate()` is the single swappable call that lights up that path; the offline demo never calls it, so output stays reproducible.

SDK names and model ids move fast; check current docs. Only `generate()` (and the stdio transport in Step 6) would need edits to go live.

In [ ]:
def generate(prompt):
    """REAL path: a hosted LLM controller is handed the discovered tool schemas. Unused offline."""
    from openai import OpenAI
    client = OpenAI()                               # reads OPENAI_API_KEY
    resp = client.chat.completions.create(
        model="gpt-4o-mini",                        # a small chat model; check names
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM would be handed the discovered schemas. "
          "Falling through to deterministic logic for reproducibility.")
else:
    print("[controller] no OPENAI_API_KEY; the host drives the in-process shim (offline default)")

## Demo - the protocol on the wire

Everything below runs fully offline through the in-process JSON-RPC shim. The demo has four sections, then the illustrative stdio transcript: (1) the `initialize` handshake plus `tools/list` discovery; (2) the three primitives `tools/call` / `resources/read` / `prompts/get`; (3) the discovered schema driving Part 1's validator; (4) multiplexing two servers into one palette and a multi-hop across both.

We start with the banner and the handshake. `host.connect(SUPPORT)` sends two frames and reads two back. The first pair is `initialize` (id 1): the host announces its protocol version and `clientInfo`; the server replies with its own `protocolVersion`, `serverInfo`, and the negotiated `capabilities`. The second pair is `tools/list` (id 2): the server returns the tool schemas, and the host folds them into its palette. After this, the host has DISCOVERED `['search_policy', 'process_refund']` without ever importing them. The schemas it now holds were never hardcoded; they arrived on the wire.

In [ ]:
bar = "=" * 72
print(bar)
print("TOOLS AS A PROTOCOL  -  a minimal MCP server and host by hand")
print(bar)
print("[transport] in-process JSON-RPC shim (default): the host calls server.handle()")
print("directly, but every frame below is a real JSON-RPC message printed verbatim.")
if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM would be handed the discovered schemas. "
          "Falling through to deterministic logic for reproducibility.")

host = MCPHost()

print("\n" + "-" * 72)
print("1) INITIALIZE handshake + tools/list discovery (the JSON-RPC wire frames).")
print("-" * 72)
host.connect(SUPPORT)
print(f"    [host] discovered tools: {list(host.palette)}")

### 2) The three primitives: tools, resources, prompts

Now exercise all three server primitives over the same shim. `tools/call` (id 3) runs `search_policy` and gets the refund-window text back as MCP content. `resources/read` (id 4) fetches the read-only refund policy file by URI. `prompts/get` (id 5) retrieves the named `refund_decision` template. Tools DO things; resources are read-only data the host can pull in as context; prompts are reusable templates the server ships. Together they are the full MCP server surface, and a host can use any of the three from any server that advertises them.

In [ ]:
print("\n" + "-" * 72)
print("2) THE THREE PRIMITIVES: tools, resources, prompts.")
print("-" * 72)
print("  tools/call:")
out = host.call_tool("search_policy", {"query": "refund window"})
print(f"    [host] result: {out}")
print("  resources/read:")
res = host._rpc(SUPPORT, "resources/read", {"uri": "file:///policies/refund.md"})
print(f"    [host] resource text: {res['contents'][0]['text']!r}")
print("  prompts/get:")
pr = host._rpc(SUPPORT, "prompts/get", {"name": "refund_decision"})
print(f"    [host] prompt template: {pr['messages'][0]['content']['text']!r}")

### 3) The discovered schema drives the controller (Part 1's validator, over the wire)

Here is the through-line of the whole part. We pull `process_refund`'s schema out of the palette (where it landed via `tools/list`) and feed it to `validate_against_schema`, the Part 1 validator, unchanged. A well-formed call passes; a missing required `amount` is rejected; a string where a number is required is rejected. The validator never saw a hardcoded dict: the schema it enforces arrived from the server over JSON-RPC. Part 1's guarantee, now provided over MCP for tools the host did not ship with.

In [ ]:
print("\n" + "-" * 72)
print("3) THE DISCOVERED SCHEMA DRIVES THE CONTROLLER (Part 1's validator, over the wire).")
print("-" * 72)
schema = host.palette["process_refund"]["schema"]
for args in ({"order_id": "ORD-3300", "amount": 180.0},
             {"order_id": "ORD-3300"},
             {"order_id": "ORD-3300", "amount": "lots"}):
    ok, err = validate_against_schema(args, schema)
    print(f"    validate {json.dumps(args)} -> {'OK' if ok else 'REJECTED: ' + err}")
print("    The schema was not hardcoded; it arrived from tools/list. Part 1's guarantee, over MCP.")

### 4) Multiplexing: one host, many servers, one palette

A host is not limited to one server. Connect a quiet host to BOTH the support and catalog servers, and its palette is the union of everything they advertise: `search_policy` and `process_refund` from support, `search_products` from catalog. To the controller it is a single flat action space; the host transparently routes each call to the server that owns the tool. The multi-hop below (two `search_products` calls answered by the catalog server) runs on the same palette where `process_refund` from the support server is also available. The agent's action space is now assembled at RUN time from whatever servers it connected to, the exact opposite of Part 1's author-time dict.

In [ ]:
print("\n" + "-" * 72)
print("4) MULTIPLEXING: connect a second server; the palette is the union of both.")
print("-" * 72)
quiet = MCPHost(verbose=False)         # connect quietly, then show the merged palette
quiet.connect(SUPPORT)
quiet.connect(CATALOG)
print(f"    [host] palette across 2 servers:")
for name, meta in quiet.palette.items():
    print(f"      {name:<16} from {meta['server']}")
print("    A multi-hop task now uses tools from different servers transparently:")
hop1 = quiet.call_tool("search_products", {"query": "who acquired Acme"})
hop2 = quiet.call_tool("search_products", {"query": "Globex earbuds warranty"})
print(f"      search_products (catalog-server) -> {hop1}")
print(f"      search_products (catalog-server) -> {hop2}")
print(f"      process_refund  (support-server) available on the same palette")

## Step 6 - The real transport: server as a subprocess over stdio (illustrative)

The verified run above used the in-process shim: a direct `server.handle()` call with real frames. The REAL deployment runs the server as a separate OS process and talks to it over stdio: the host writes JSON-RPC to the subprocess's stdin and reads responses off its stdout. Two local processes, a pipe between them, no network.

The cell below PRINTS that transcript; it does not execute it. It is clearly labeled "not executed here" because spawning a subprocess is not byte-reproducible the way the shim is, and this notebook's bar is determinism. The point to carry away: the frames crossing the pipe are the same frames you saw above. Only the delivery mechanism changes, swap `server.handle(req)` in `_rpc()` for a stdin write plus a stdout read, and the entire protocol layer is untouched.

In [ ]:
print("\n" + bar)
print("THE REAL TRANSPORT (illustrative, not executed here): server as a subprocess over")
print("stdio. Two local processes, no network. Shown for reference; the verified run above")
print("uses the in-process shim.")
print(bar)
print("    $ python mcp_server.py            # process A: reads JSON-RPC from stdin")
print('    host -> server (stdin):  {"jsonrpc":"2.0","id":1,"method":"initialize",...}')
print('    server -> host (stdout): {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":...}}')
print('    host -> server (stdin):  {"jsonrpc":"2.0","id":2,"method":"tools/list"}')
print('    server -> host (stdout): {"jsonrpc":"2.0","id":2,"result":{"tools":[...]}}')
print("    (same frames as above, now crossing a pipe between two OS processes)")

print("\n" + bar)
print("Done. Tools are no longer a hardcoded dict:")
print("  - a SERVER advertises tools/resources/prompts; a HOST discovers them over JSON-RPC")
print("  - the initialize handshake negotiates the protocol version + capabilities")
print("  - discovered inputSchemas feed Part 1's validator/controller, unchanged")
print("  - one host MULTIPLEXES many servers into a single, run-time-assembled palette")
print(bar)

## Sidebar - Skills and progressive tool disclosure

MCP discovery has a scaling problem hiding in it. A host connected to many servers can be handed HUNDREDS of tools, and stuffing every schema into the prompt burns tokens (the transcript-economics lever from Part 11) and dilutes the model's attention. The fix is **progressive tool disclosure**: do not put all the tools in the prompt at once. Disclose them in stages.

- **Names first.** List just tool names (and a one-line description) so the model knows what exists.
- **Schema on demand.** Fetch a tool's full `inputSchema` only when the model reaches for it. That is exactly `tools/list` for the cheap summary and a per-tool schema lookup when needed.
- **Skills.** A related idea: bundle a set of tools, instructions, and resources behind a single named capability the model loads only when relevant, so a server can advertise one "skill" rather than twenty raw tools.

This pairs naturally with MCP's discovery calls: the protocol already separates "what exists" (`tools/list`) from "how to call this one" (the schema), so disclosing progressively is a host-side policy on top of the same wire. It is the action-space analogue of Part 7's compaction: keep the working context small by loading detail only when it is reached for.

## Wrap-up: tools now speak a protocol

The frontier track opens by cutting the last cord that kept the agent self-contained. Its tools were a hardcoded Python dict; now they are discovered, schema-described capabilities served over a wire protocol.

- **A server and a host, by hand.** The server answers JSON-RPC: `initialize` (protocol + capability negotiation), `tools/list`, `tools/call`, and the three primitives `tools` / `resources` / `prompts`. The host does the handshake, discovers the tools, and routes calls.
- **Discovery feeds Part 1's validator, unchanged.** The `inputSchema` arrives over `tools/list` and goes straight into the same validator that guarded the hardcoded dict. A JSON Schema is just data; the validator does not care where it came from.
- **One host multiplexes many servers.** The palette is the union of every connected server, assembled at run time, used transparently across a multi-hop.
- **Honest transport.** The verified default is an in-process JSON-RPC shim printing real frames; the production path is the server as a subprocess over stdio, shown as a clearly labeled illustrative transcript. Swapping one for the other touches only how a frame is delivered, not the protocol.
- **Progressive disclosure / skills.** When a host holds hundreds of discovered tools, disclose names first and full schemas on demand, the action-space cousin of Part 7's compaction.

**Part 13 - code execution and the sandbox.** A tool call over MCP still runs code that some process must execute. Part 13 asks where: instead of one fixed tool per capability, give the agent a sandbox it can write and run code in, and confront the safety question that opens, running model-written code without letting it run away with the machine.